In [0]:
CATALOG = "github_pipeline_catalog"
SCHEMA = "github_analytics"

spark.sql(
    f"USE CATALOG `{CATALOG}`"
)

spark.sql(
    f"USE SCHEMA `{SCHEMA}`"
)

print(CATALOG, SCHEMA)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
`{CATALOG}`.`{SCHEMA}`.`gold_dim_repository`
(
    repo_id BIGINT,
    repository_name STRING,
    full_name STRING,
    owner_login STRING,
    language STRING,
    default_branch STRING,
    created_at TIMESTAMP,
    archived BOOLEAN,
    fork BOOLEAN,

    record_hash STRING,
    source_batch_id STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

print("gold_dim_repository created")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
`{CATALOG}`.`{SCHEMA}`.`gold_dim_date`
(
    date_key INT,
    full_date DATE,
    year INT,
    quarter INT,
    month INT,
    month_name STRING,
    week_of_year INT,
    day_of_month INT,
    day_of_week INT,
    day_name STRING,
    is_weekend BOOLEAN
)
USING DELTA
""")

In [0]:
from pyspark.sql import functions as F

In [0]:
date_table = (
    f"{CATALOG}.{SCHEMA}.gold_dim_date"
)

if spark.table(date_table).count() == 0:

    dates = spark.sql("""
        SELECT explode(
            sequence(
                DATE('2008-01-01'),
                DATE('2035-12-31'),
                INTERVAL 1 DAY
            )
        ) AS full_date
    """)

    dim_date = (
        dates
        .withColumn(
            "date_key",
            F.date_format(
                "full_date",
                "yyyyMMdd"
            ).cast("int")
        )
        .withColumn(
            "year",
            F.year("full_date")
        )
        .withColumn(
            "quarter",
            F.quarter("full_date")
        )
        .withColumn(
            "month",
            F.month("full_date")
        )
        .withColumn(
            "month_name",
            F.date_format(
                "full_date",
                "MMMM"
            )
        )
        .withColumn(
            "week_of_year",
            F.weekofyear("full_date")
        )
        .withColumn(
            "day_of_month",
            F.dayofmonth("full_date")
        )
        .withColumn(
            "day_of_week",
            F.dayofweek("full_date")
        )
        .withColumn(
            "day_name",
            F.date_format(
                "full_date",
                "EEEE"
            )
        )
        .withColumn(
            "is_weekend",
            F.dayofweek(
                "full_date"
            ).isin(1, 7)
        )
        .select(
            "date_key",
            "full_date",
            "year",
            "quarter",
            "month",
            "month_name",
            "week_of_year",
            "day_of_month",
            "day_of_week",
            "day_name",
            "is_weekend"
        )
    )

    (
        dim_date
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(date_table)
    )

print(
    "Date rows:",
    spark.table(date_table).count()
)

In [0]:
from pyspark.sql import functions as F

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
`{CATALOG}`.`{SCHEMA}`.
`gold_fact_repository_snapshot`
(
    repo_id BIGINT,
    date_key INT,

    stars BIGINT,
    forks BIGINT,
    open_issues BIGINT,

    star_change BIGINT,
    fork_change BIGINT,
    open_issue_change BIGINT,

    record_hash STRING,
    source_batch_id STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
`{CATALOG}`.`{SCHEMA}`.
`gold_fact_repository_activity`
(
    repo_id BIGINT,
    date_key INT,

    commit_count BIGINT,
    active_commit_authors BIGINT,

    issues_opened BIGINT,
    issues_closed BIGINT,
    issue_resolution_hours_sum DOUBLE,

    pull_requests_created BIGINT,
    pull_requests_merged BIGINT,
    pr_merge_hours_sum DOUBLE,

    record_hash STRING,
    refresh_batch_id STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
`{CATALOG}`.`{SCHEMA}`.
`gold_fact_pipeline_execution`
(
    run_id STRING,
    date_key INT,

    layer STRING,
    dataset STRING,
    status STRING,
    source_load_type STRING,
    execution_mode STRING,

    rows_read BIGINT,
    rows_inserted BIGINT,
    rows_updated BIGINT,
    rows_quarantined BIGINT,

    execution_duration_seconds BIGINT,

    error_message STRING,

    record_hash STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

In [0]:
display(
    spark.sql(
        f"SHOW TABLES IN "
        f"`{CATALOG}`.`{SCHEMA}`"
    )
)